In [7]:
import os
import re
import time
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class NBASemanticVectorPipeline:
    def __init__(self, target_players_df_path, output_csv_name="nlp_model_scores.csv", credits_csv_name="articles_credited.csv"):
        self.output_csv = output_csv_name
        self.credits_csv = credits_csv_name
        self.api_url = "https://en.wikipedia.org/w/api.php"
        self.headers = {"User-Agent": "NBASemanticBot/2.0 (sportsanalyticsproject@example.com)"}
        
        if os.path.exists(target_players_df_path):
            master_df = pd.read_csv(target_players_df_path)
            self.target_players = master_df['Player_Name'].dropna().unique().tolist()
            print(f"🎯 Master roster loaded. Tracking {len(self.target_players)} entries.")
        else:
            raise FileNotFoundError(f"Could not find your master CSV file at: {target_players_df_path}")

        # Core semantic anchor phrases representing your exact model traits
        self.concepts = {
            "Respect_Score": [
                "greatest basketball player of all time legend revered goat prestige iconic legacy",
                "dominant elite status hall of fame accolades unmatched peak supreme fear praise"
            ],
            "Failings_Score": [
                "career failings struggles collapse choked flaw weakness mistake breakdown disappointment",
                "criticized for missing clutch shots turnovers liability losing series internal downfall"
            ],
            "Positive_Attitude_Score": [
                "positive attitude leadership inspiring locker room presence unselfish teammate mentor",
                "charity outreach dedicated work ethic professional supportive captain champion chemistry"
            ],
            "Negative_Attitude_Score": [
                "negative attitude toxic arrogant selfish complaining frustrated feud distraction drama",
                "demanded trade clashed with coach team distraction arguments tension technical fouls"
            ]
        }

    def fetch_wikipedia_text_and_url(self, player_name):
        """
        First searches Wikipedia to get the corrected page title (e.g., converts 'Steph Curry' -> 'Stephen Curry'),
        then extracts the full, rich career biography text safely.
        """
        search_params = {
            "action": "query", "list": "search", "srsearch": player_name, "format": "json", "srlimit": 1
        }
        
        try:
            search_res = requests.get(self.api_url, params=search_params, headers=self.headers, timeout=10)
            if search_res.status_code == 200:
                search_data = search_res.json()
                search_results = search_data.get("query", {}).get("search", [])
                if not search_results:
                    return "", ""
                
                corrected_title = search_results[0]["title"]
            else:
                return "", ""
                
            time.sleep(1) # Intentional delay to protect against network connection blocks
            parse_params = {
                "action": "query", "format": "json", "titles": corrected_title,
                "prop": "extracts|info", "exintro": 0, "explaintext": 1, "inprop": "url"
            }
            
            response = requests.get(self.api_url, params=parse_params, headers=self.headers, timeout=10)
            if response.status_code == 200:
                pages = response.json().get("query", {}).get("pages", {})
                for page_id, page_data in pages.items():
                    if page_id != "-1":
                        return page_data.get("extract", ""), page_data.get("fullurl", "")
        except Exception as e:
            print(f"⚠️ Connection glitch during fetch for {player_name}: {e}")
            
        return "", ""

    def extract_player_context(self, player_name, text):
        """Extracts comprehensive text segments centered around the player's name variants."""
        if not text or len(text.strip()) < 100:
            return ""

        sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text.strip())
        relevant_segments = []
        
        name_parts = player_name.split()
        last_name = name_parts[-1] if len(name_parts) > 1 else ""
        first_name = name_parts[0]
        
        aliases = [player_name.lower()]
        if len(last_name) > 3: aliases.append(last_name.lower())
        if len(first_name) > 3: aliases.append(first_name.lower())

        for idx, sentence in enumerate(sentences):
            sentence_lower = sentence.lower()
            if any(alias in sentence_lower for alias in aliases):
                start = max(0, idx - 2)
                end = min(len(sentences), idx + 3)
                relevant_segments.append(" ".join(sentences[start:end]))

        return " ".join(relevant_segments)

    def calculate_semantic_similarity(self, context_text):
        """Maps content vectors into cosine mathematical angles for true meaning parsing."""
        scores = {}
        for score_name, anchor_phrases in self.concepts.items():
            documents = [context_text] + anchor_phrases
            vectorizer = TfidfVectorizer(stop_words='english')
            tfidf_matrix = vectorizer.fit_transform(documents)
            similarity_matrix = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])
            scores[score_name] = round(float(similarity_matrix.mean()) * 100, 3)
        return scores

    def run_pipeline(self):
        all_results = []
        
        for player in self.target_players:
            wiki_text, wiki_url = self.fetch_wikipedia_text_and_url(player)
            context_chunk = self.extract_player_context(player, wiki_text)
            
            if context_chunk and len(context_chunk.split()) > 15:
                print(f"🧠 Vectorizing semantic narratives for: {player}...")
                scores = self.calculate_semantic_similarity(context_chunk)
                
                player_result = {
                    "Player_Name": player,
                    "Context_Length": len(context_chunk),
                    "Respect_Score": scores["Respect_Score"],
                    "Failings_Score": scores["Failings_Score"],
                    "Positive_Attitude_Score": scores["Positive_Attitude_Score"],
                    "Negative_Attitude_Score": scores["Negative_Attitude_Score"]
                }
                all_results.append(player_result)
                
                new_credit = pd.DataFrame([{"Player_Name": player, "Source_URL": wiki_url}])
                if os.path.exists(self.credits_csv):
                    pd.concat([pd.read_csv(self.credits_csv), new_credit]).drop_duplicates().to_csv(self.credits_csv, index=False)
                else:
                    new_credit.to_csv(self.credits_csv, index=False)
            else:
                print(f"❌ Skipping {player} (Insufficient media text or missing search result found)")
                
            time.sleep(1.5)

        if all_results:
            new_df = pd.DataFrame(all_results)
            if os.path.exists(self.output_csv):
                existing_df = pd.read_csv(self.output_csv)
                existing_df.set_index("Player_Name", inplace=True)
                new_df.set_index("Player_Name", inplace=True)
                final_df = new_df.combine_first(existing_df).reset_index()
            else:
                final_df = new_df
                
            final_df.to_csv(self.output_csv, index=False)
            print(f"\n💾 Pipeline run complete! Files generated:\n📊 Metrics: '{self.output_csv}'\n🔗 Sources: '{self.credits_csv}'")
        else:
            print("❌ Pipeline execution halted. All text fetches returned blank values.")

if __name__ == "__main__":
    master_file = os.path.join('..', 'Data Collection', 'nba_top_100_master.csv')
    
    engine = NBASemanticVectorPipeline(
        target_players_df_path=master_file,
        output_csv_name="nlp_model_scores.csv",
        credits_csv_name="articles_credited.csv"
    )
    engine.run_pipeline()


🎯 Master roster loaded. Tracking 114 entries.
🧠 Vectorizing semantic narratives for: Michael Jordan...
🧠 Vectorizing semantic narratives for: LeBron James...
🧠 Vectorizing semantic narratives for: Kareem Abdul-Jabbar...
🧠 Vectorizing semantic narratives for: Bill Russell...
🧠 Vectorizing semantic narratives for: Magic Johnson...
🧠 Vectorizing semantic narratives for: Wilt Chamberlain...
🧠 Vectorizing semantic narratives for: Shaquille O'Neal...
🧠 Vectorizing semantic narratives for: Tim Duncan...
🧠 Vectorizing semantic narratives for: Larry Bird...
🧠 Vectorizing semantic narratives for: Kobe Bryant...
🧠 Vectorizing semantic narratives for: Hakeem Olajuwon...
🧠 Vectorizing semantic narratives for: Stephen Curry...
🧠 Vectorizing semantic narratives for: Oscar Robertson...
🧠 Vectorizing semantic narratives for: Kevin Durant...
🧠 Vectorizing semantic narratives for: Julius Erving...
🧠 Vectorizing semantic narratives for: Jerry West...
🧠 Vectorizing semantic narratives for: Karl Malone...
🧠

In [10]:
import pandas as pd
import os

def rank_player_nlp_scores(input_csv="nlp_model_scores.csv", output_csv="nlp_model_scores_ranked.csv"):
    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"Could not locate '{input_csv}'. Ensure your semantic engine script has run successfully first.")
        
    # Read the data
    df = pd.read_csv(input_csv)
    
    # Apply the semantic algorithm formula
    df['Overall_NLP_Score'] = (
        (df['Respect_Score'] + df['Positive_Attitude_Score']) - 
        (df['Failings_Score'] + df['Negative_Attitude_Score'])
    )
    
    # Clean up decimal precision to 3 places
    df['Overall_NLP_Score'] = df['Overall_NLP_Score'].round(3)
    
    # Sort values descending by the net score
    df_ranked = df.sort_values(by='Overall_NLP_Score', ascending=False).reset_index(drop=True)
    
    # Generate the official narrative standing tier ranks
    df_ranked['NLP_Rank'] = df_ranked.index + 1
    
    # Reorder columns to place key insights up front
    ordered_cols = [
        'NLP_Rank', 'Player_Name', 'Overall_NLP_Score', 
        'Respect_Score', 'Failings_Score', 'Positive_Attitude_Score', 'Negative_Attitude_Score'
    ]
    final_df = df_ranked[ordered_cols]
    
    # Overwrite/save the file with rankings included
    final_df.to_csv(output_csv, index=False)
    
    print(f"📊 Calculations complete! Successfully ranked {len(final_df)} players.")
    print(f"💾 File updated and exported to: '{output_csv}'")
    print("\n👑 Top 5 Narrative Leaders Preview:")
    print(final_df.head(5).to_string(index=False))

if __name__ == "__main__":
    rank_player_nlp_scores()

📊 Calculations complete! Successfully ranked 115 players.
💾 File updated and exported to: 'nlp_model_scores_ranked.csv'

👑 Top 5 Narrative Leaders Preview:
 NLP_Rank     Player_Name  Overall_NLP_Score  Respect_Score  Failings_Score  Positive_Attitude_Score  Negative_Attitude_Score
        1 Spencer Haywood             11.600          9.752           0.000                    1.848                    0.000
        2  Adrian Dantley             11.214         10.920           0.000                    1.774                    1.480
        3   Tracy McGrady             10.249         10.224           0.982                    1.760                    0.753
        4     Jerry Lucas              9.913          9.608           0.000                    0.956                    0.651
        5    Bernard King              9.682          9.682           0.928                    0.928                    0.000
